# 03 — Delta Lake: ACID + time travel

Delta es un formato de tabla que vive **encima de Parquet** y le
agrega:

- Transacciones ACID (lecturas consistentes durante escrituras)
- Time travel (volver a versiones anteriores)
- MERGE / UPDATE / DELETE (UPSERT, casi magia)
- Schema evolution (agregar columnas sin romper)

## Setup

Delta no viene en la imagen de Jupyter. Lo agregamos a `EXTRA_PYTHON_PACKAGES`
y como JAR al classpath. La forma más simple es agregar a tu compose:

```yaml
environment:
  - EXTRA_PYTHON_PACKAGES=delta-spark==3.0.0
  - SPARK_OPTS=--packages io.delta:delta-spark_2.12:3.0.0
```

Alternativa, dentro del notebook, agregar el JAR con `spark.jars.packages`.

In [ ]:
import os
# Si no configuraste EXTRA_PYTHON_PACKAGES, cargá el JAR dinámicamente:
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages io.delta:delta-spark_2.12:3.0.0 "
    "--conf spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension "
    "--conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog "
    "pyspark-shell"
)

from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("02_03_delta_lake")
    .master("spark://spark-master:7077")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

## 1. Escritura inicial

In [ ]:
DELTA_PATH = "/home/jovyan/work/data/sample/ventas/delta"

df = spark.read.parquet("/home/jovyan/work/data/sample/ventas/parquet_limpio")
df.limit(10_000).write.format("delta").mode("overwrite").save(DELTA_PATH)
print("Tabla Delta escrita en", DELTA_PATH)

## 2. Time travel

In [ ]:
delta_df = spark.read.format("delta").load(DELTA_PATH)
print("Versión 0:", delta_df.count())

# Appendeamos más datos
df.limit(5_000).write.format("delta").mode("append").save(DELTA_PATH)
print("Versión 1 (después de append):", spark.read.format("delta").load(DELTA_PATH).count())

# Volvemos a la versión 0
v0 = spark.read.format("delta").option("versionAsOf", 0).load(DELTA_PATH)
print("Leyendo versionAsOf=0:", v0.count())

## 3. UPSERT con MERGE

La operación que cambió la ingeniería de datos. En vez de:
1. Borrar todo y reescribir
2. Detectar diff en Python y hacer un patch raro

Hacés `MERGE INTO`:

In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, DELTA_PATH)

# Subimos un precio nuevo para un id específico
updates = spark.createDataFrame(
    [(1, 99999.99), (2, 88888.88)],
    ["id", "precio"],
)

(
    delta_table.alias("t")
    .merge(
        updates.alias("u"),
        "t.id = u.id",
    )
    .whenMatchedUpdate(set={"precio": "u.precio"})
    .whenNotMatchedInsert(values={"id": "u.id", "precio": "u.precio", "ciudad": F.lit("Bogotá")})
    .execute()
)

print("MERGE ejecutado")
spark.read.format("delta").load(DELTA_PATH).filter("id IN (1, 2)").show()

## 4. Historia de la tabla

In [ ]:
delta_table.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

## ✅ Checkpoint

- [x] Sé escribir y leer tablas Delta.
- [x] Sé hacer time travel.
- [x] Sé hacer UPSERT con MERGE.
- [x] Sé inspeccionar la historia.

**Siguiente módulo**: `03_streaming/` — Structured Streaming sobre las
mismas tablas Delta.